In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

driver = webdriver.Chrome()
driver.maximize_window()
wait = WebDriverWait(driver, 10)

In [ ]:
try:
    driver.get("http://localhost:5173/")
    driver.execute_script("window.localStorage.clear(); window.sessionStorage.clear();")
    driver.get("http://localhost:5173/login")

    wait.until(EC.presence_of_element_located((By.ID, "username")))
    driver.find_element(By.ID, "username").send_keys("shawon@gmail.com")
    driver.find_element(By.ID, "password").send_keys("12345678")
    driver.find_element(By.ID, "sign-in-btn").click()
    time.sleep(3)

    # Open Reports via the real sidebar (label verified in AppShell.jsx)
    wait.until(EC.element_to_be_clickable((By.XPATH, "//aside//button[contains(., 'Reports')]"))).click()
    wait.until(EC.visibility_of_element_located((By.XPATH, "//button[text()='30 Days']")))
    print("PASS: Reports page opened")

    # Wait for loading to finish, then check the real content (verified in ReportsPage.jsx)
    wait.until(EC.invisibility_of_element_located((By.XPATH, "//*[text()='Loading reports...']")))
    time.sleep(2)
    body = driver.find_element(By.TAG_NAME, "body").text
    cards = [t for t in ["Revenue", "Sales", "Items Sold", "Profit"] if t in body]
    sections = [t for t in ["Daily Sales Trend", "Stock Report", "Top Selling Products"] if t in body]
    charts = driver.find_elements(By.XPATH, "//div[contains(@class, 'recharts-wrapper')]")
    empty = [t for t in ["No sales in this period", "No product sales in this period"] if t in body]
    print("Cards:", cards, "| Sections:", sections, "| Charts:", len(charts), "| Empty states:", empty)
    assert cards or sections or charts or empty, "No report content, chart, or empty state rendered."
    print("PASS: Report content rendered")
except Exception as e:
    print("FAIL:", e)
    driver.save_screenshot("36_reports_FAIL.png")
finally:
    driver.quit()